# AMBER RAG LLM Support System

Notebook generated from `testRAG.py`.

## Setup

This notebook mirrors the logic in `testRAG.py` but is organized into runnable cells.
Run cells top-to-bottom, then use the **Ask a question** section at the end.

In [1]:
import os
import re
import shutil
import textwrap
import traceback
import subprocess
import database_menu
from pathlib import Path
from typing import List, Tuple, Dict, Any
from database_menu import AmberChromaAPI, EMBEDDER


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


----------------------------
### Config 
----------------------------

In [2]:
DEFAULT_DB_PATH = os.getenv("CHROMA_DIR", "/opt/chromadb/data/amber_chroma_db")
COLLECTION_NAME = os.getenv("COLLECTION_NAME", "amber_messages")

N_CANDIDATES = int(os.getenv("N_CANDIDATES", "30"))
TOP_K = int(os.getenv("TOP_K", "5"))
MIN_DOC_CHARS = int(os.getenv("MIN_DOC_CHARS", "200"))

WEIGHT = float(os.getenv("COLLECTION_WEIGHT", "1.0"))

# LLM / Ollama
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1")
OLLAMA_TIMEOUT = int(os.getenv("OLLAMA_TIMEOUT", "180"))


----------------------------
### Tokenization Helpers
----------------------------

In [3]:
_word_re = re.compile(r"[a-zA-Z0-9_]+")

def tokenize(s: str) -> List[str]:
    return [t.lower() for t in _word_re.findall(s or "")]

def keyword_overlap_score(query: str, doc: str) -> float:
    q = tokenize(query)
    d = tokenize(doc)
    if not q or not d:
        return 0.0
    overlap = set(q).intersection(set(d))
    score = 0.0
    for tok in overlap:
        score += 1.0 + min(len(tok), 12) / 12.0
    return score / (len(set(q)) ** 0.5)

def best_window_snippet(query: str, text: str, window_chars: int = 700) -> str:
    if not text:
        return ""
    parts = re.split(r"(?<=[\.\?\!])\s+|\n+", text)
    parts = [p.strip() for p in parts if p.strip()]
    q_tokens = set(tokenize(query))
    scored = []
    for p in parts:
        overlap = len(q_tokens.intersection(set(tokenize(p))))
        if overlap:
            scored.append((overlap, p))
    if not scored:
        return text[:window_chars]
    scored.sort(key=lambda x: x[0], reverse=True)
    snippet = " ".join(p for _, p in scored[:6])
    if len(snippet) > window_chars:
        snippet = snippet[:window_chars] + "…"
    return snippet

def format_similarity(space: str, dist: float) -> str:
    # database_menu uses cosine space by default
    if space == "cosine":
        return f"{1 - dist:.4f} (cosine-sim approx)"
    return f"{dist:.4f} (distance; lower is better)"


----------------------------
### Cross-Encoder reranker
----------------------------

In [4]:
_RERANKER = None

def get_reranker():
    global _RERANKER
    if _RERANKER is not None:
        return _RERANKER
    try:
        from sentence_transformers import CrossEncoder
        model_name = os.getenv("RERANK_MODEL", "cross-encoder/ms-marco-MiniLM-L-6-v2")
        print(f"Loading reranker model: {model_name}")
        _RERANKER = CrossEncoder(model_name)
        return _RERANKER
    except Exception:
        print("⚠️ Failed to load cross-encoder (fallback to keyword overlap).")
        traceback.print_exc()
        _RERANKER = None
        return None

def try_cross_encoder_rerank(query: str, docs: List[str]) -> List[float]:
    reranker = get_reranker()
    if reranker is None:
        return []
    try:
        pairs = [(query, d) for d in docs]
        scores = reranker.predict(pairs)
        return [float(s) for s in scores]
    except Exception:
        traceback.print_exc()
        return []


----------------------------
### Ollama Helpers
----------------------------

In [5]:
def find_ollama_executable() -> str | None:
    p = shutil.which("ollama")
    if p:
        return p
    env = os.getenv("OLLAMA_PATH")
    if env and Path(env).exists():
        return env
    if os.name == "nt":
        candidates = [
            Path(os.getenv("LOCALAPPDATA", "")) / "Programs" / "Ollama" / "ollama.exe",
            Path("C:/Program Files/Ollama/ollama.exe"),
            Path("C:/Program Files (x86)/Ollama/ollama.exe"),
        ]
        for c in candidates:
            if c.exists():
                return str(c)
    return None

def generate_with_ollama(question: str, context: str, source_url: str | None = None) -> str:
    ollama_exec = find_ollama_executable()
    if not ollama_exec:
        print("⚠️ Ollama executable not found.")
        return "(Ollama not found — returning retrieved context)\n\n" + context

    print(f"🦙 Using Ollama executable: {ollama_exec}\n")

    source_block = ""
    if source_url:
        source_block = f"\nMost relevant source:\n{source_url}\n"

    prompt = textwrap.dedent(f"""\
        You are a helpful assistant.
        Answer the question using ONLY the context provided.
        If a source link is provided and relevant, cite it at the end as:
        Source: <url>

        Question:
        {question}

        Context:
        {context}
        {source_block}

        Answer:
    """)

    proc = subprocess.run(
        [ollama_exec, "run", OLLAMA_MODEL],
        input=prompt.encode(),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        timeout=OLLAMA_TIMEOUT,
    )

    out = proc.stdout.decode(errors="replace").strip()
    err = proc.stderr.decode(errors="replace").strip()

    if proc.returncode != 0:
        return f"Ollama failed.\nStderr:\n{err}\nStdout:\n{out}"

    if source_url and "source:" not in out.lower():
        out += f"\nSource: {source_url}"

    return out


----------------------------
### Chroma Query Across Collection
----------------------------

In [6]:
def get_space(coll) -> str:
    try:
        meta = coll.metadata or {}
        return meta.get("hnsw:space", "unknown")
    except Exception:
        return "unknown"

def query_collection(coll, question: str, n: int, collection_label: str, weight: float) -> List[Tuple[float, Dict[str, Any]]]:
    """
    Returns list of (weight, item_dict).
    item_dict contains doc/meta/dist/id/collection for downstream rerank + printing.
    """
    q_emb = EMBEDDER.encode(question).tolist()

    res = coll.query(
        query_embeddings=[q_emb],
        n_results=n,
        include=["documents", "metadatas", "distances"],
    )

    ids = res.get("ids", [[]])[0]
    docs = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]

    out = []
    for id_, doc, meta, dist in zip(ids, docs, metas, dists):
        if not doc:
            continue
        doc = str(doc)
        if len(doc) < MIN_DOC_CHARS:
            continue
        item = {
            "collection": collection_label,
            "id": id_,
            "doc": doc,
            "meta": meta or {},
            "dist": float(dist),
        }
        out.append((weight, item))
    return out


----------------------------
### RAG Flow
----------------------------

In [7]:

def open_collections(db_path: str, collection: str):
    # Use AmberChromaAPI to ensure db folder exists and to reuse its client wiring
    col_api = AmberChromaAPI(db_path=db_path, collection_name=collection)
    return col_api

def rag_answer_flow(collection_Name, question: str):
    space_col = get_space(collection_Name)

    # Pull candidates
    candidates = query_collection(collection_Name, question, N_CANDIDATES, "tutorials", WEIGHT)

    if not candidates:
        print("⚠️ No candidates found (docs too short or empty).")
        return

    # Rerank all together
    docs_for_rerank = [it["doc"] for _, it in candidates]
    ce_scores = try_cross_encoder_rerank(question, docs_for_rerank)

    if ce_scores and len(ce_scores) == len(candidates):
        merged = []
        for (w, item), ce in zip(candidates, ce_scores):
            merged.append((float(ce) * float(w), float(ce), item))
        merged.sort(key=lambda x: x[0], reverse=True)
        rerank_label = "cross-encoder (weighted by collection)"
    else:
        merged = []
        for (w, item) in candidates:
            kw = keyword_overlap_score(question, item["doc"])
            merged.append((kw * float(w), kw, item))
        merged.sort(key=lambda x: x[0], reverse=True)
        rerank_label = "keyword-overlap fallback (weighted by collection)"

    print(f"✅ Rerank method: {rerank_label}")
    print(f"🔎 Candidates: {len(merged)} | Showing top {TOP_K}")

    # Print top-k
    for i, (weighted, rawscore, item) in enumerate(merged[:TOP_K], 1):
        meta = item["meta"] or {}
        title = meta.get("title") or meta.get("subject") or "(no title/subject)"
        url = meta.get("url") or "(no url)"
        snippet = best_window_snippet(question, item["doc"], 750)

        sim_display = format_similarity(
            space_col,
            item["dist"],
        )

        print(f"\n{i}. [{item['collection']}] {title}")
        print(f"   id: {item['id']}")
        print(f"   url: {url}")
        print(f"   vector: {sim_display}")
        print(f"   rerank_score: {rawscore:.4f} | weighted: {weighted:.4f}")
        print(f"   best_snippet: {snippet}")

    # Build context from top-k
    context = "\n\n---\n\n".join(
        best_window_snippet(question, item["doc"], 700)
        for _, _, item in merged[:TOP_K]
    )

    best_meta = merged[0][2].get("meta", {}) if merged else {}
    most_relevant_url = best_meta.get("url")

    print("\n================= FINAL ANSWER (LLM) =================\n")
    answer = generate_with_ollama(question, context, most_relevant_url)
    print(answer)


## Ex. Ask a question

In [8]:
from database_menu import AmberChromaAPI

# Create/open the collection
api = AmberChromaAPI(db_path=DEFAULT_DB_PATH, collection_name=COLLECTION_NAME)

# Ask your question
question = """How do you perform energy minimization in AMBER?"""

# Run the RAG pipeline
rag_answer_flow(api.collection, question)


Using local ChromaDB path: /opt/chromadb/data/amber_chroma_db
Loading reranker model: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Rerank method: cross-encoder (weighted by collection)
🔎 Candidates: 29 | Showing top 5

1. [tutorials] [AMBER] pmemd.cuda:command not found
   id: thread-1578209312
   url: (no url)
   vector: 0.2019 (cosine-sim approx)
   rerank_score: -2.0385 | weighted: -2.0385
   best_snippet: Thanking you in advance. Make sure it is in your path, by sourcing amber.sh The Amber manual has clear instructions on installing pmemd with CUDA that you can follow. > Thanking you in advance. I wanted to run  a minimization but encountered the following error while ssr.clogin72:~/amber18/bin> pmemd.cuda -O -i Min.in -o Min.out -p

2. [tutorials] [AMBER] Resolving overlapping atoms.
   id: thread-1578563652
   url: (no url)
   vector: 0.3000 (cosine-sim approx)
   rerank_score: -2.7818 | weighted: -2.7818
   best_snippet: you are just wasting computer time: does your energy not converge after > you are just wasting computer time: does your energy not converge after After adding water by addtobox  to a pDB 

--------------------
### Interactive menu
--------------------

In [9]:
def main():
    col_api = None
    db_path_display = "None"

    while True:
        print("\n========== AMBER RAG MENU ==========")
        print(f"Current database: {db_path_display}")
        print(f"Data Collection: {COLLECTION_NAME}")
        print(f"Ollama model:         {OLLAMA_MODEL}")
        print("1. Open DB")
        print("2. Add JSON to collection")
        print("3. Peek collection")
        print("4. RAG Ask (top-k + Llama answer)")
        print("5. Exit")
        print("====================================")

        choice = input("Choose an option (1-5): ").strip()

        if choice == "1":
            db_path = input(f"Database folder (default: {DEFAULT_DB_PATH}): ").strip() or DEFAULT_DB_PATH
            col_api = open_collections(db_path, COLLECTION_NAME)
            db_path_display = os.path.abspath(db_path)

            try:
                print(f"📦 '{COLLECTION_NAME}' count: {col_api.collection.count()}")
            except Exception:
                pass

        elif choice == "2":
            if col_api is None:
                print("No database is currently open. Use option 1 first.")
                continue

            path = input("Enter JSON filename: ").strip()
            col_api.add_json(path)
            print(f"Added threads from {path} into {COLLECTION_NAME}")

        elif choice == "3":
            if col_api is None:
                print("No database is currently open. Use option 1 first.")
                continue

            peeked = col_api.peek()
            docs = peeked.get("documents", [])
            metas = peeked.get("metadatas", [])
            if not docs:
                print("No entries found in this collection.")
            else:
                print(f"\nPreviewing first {min(len(docs), 5)} entries:\n")
                for meta, doc in zip(metas[:5], docs[:5]):
                    author = (meta or {}).get("author", "Unknown")
                    subject = (meta or {}).get("subject", "(no subject)")
                    print(f"[{author}] {subject}")
                    print(str(doc)[:250] + "...\n" + "-" * 60)

        elif choice == "4":
            if col_api is None:
                print("No database is currently open. Use option 1 first.")
                continue

            while True:
                q = input("\nEnter question (or 'back'): ").strip()
                if q.lower() == "back":
                    break
                rag_answer_flow(col_api.collection, q)

        elif choice == "5":
            print("Exiting.")
            break

        else:
            print("Invalid choice.")


if __name__ == "__main__":
    main()


========== AMBER RAG MENU ==========
Current database: None
Data Collection: amber_messages
Ollama model:         llama3.1
1. Open DB
2. Add JSON to collection
3. Peek collection
4. RAG Ask (top-k + Llama answer)
5. Exit


Choose an option (1-5):  1
Database folder (default: /opt/chromadb/data/amber_chroma_db):  


Using local ChromaDB path: /opt/chromadb/data/amber_chroma_db
📦 'amber_messages' count: 51

========== AMBER RAG MENU ==========
Current database: /opt/chromadb/data/amber_chroma_db
Data Collection: amber_messages
Ollama model:         llama3.1
1. Open DB
2. Add JSON to collection
3. Peek collection
4. RAG Ask (top-k + Llama answer)
5. Exit


Choose an option (1-5):  4

Enter question (or 'back'):  What is up


✅ Rerank method: cross-encoder (weighted by collection)
🔎 Candidates: 29 | Showing top 5

1. [tutorials] [AMBER] Lipid bilayer imaging problem
   id: thread-1578571087
   url: (no url)
   vector: 0.0026 (cosine-sim approx)
   rerank_score: -7.7916 | weighted: -7.7916
   best_snippet: What would be the cause and the possible solution for this What would be the cause and the possible solution for this What would be the cause and the possible solution for this

2. [tutorials] [AMBER] [ Polymer with Extra Points ]
   id: thread-1578379238
   url: (no url)
   vector: 0.0693 (cosine-sim approx)
   rerank_score: -8.2830 | weighted: -8.2830
   best_snippet: I can make polymer without extra points, which is explained at tutorial The way to do this is somewhat involved for a beginner. What you will need to do is get a prepin format file for your molecule (antechamber I can tell you then what I did so you see how > I can make polymer without extra points, which is explained at tutorial

3. [tutor

...


========== AMBER RAG MENU ==========
Current database: /opt/chromadb/data/amber_chroma_db
Data Collection: amber_messages
Ollama model:         llama3.1
1. Open DB
2. Add JSON to collection
3. Peek collection
4. RAG Ask (top-k + Llama answer)
5. Exit


Choose an option (1-5):  3



Previewing first 5 entries:

[Rui Chen] [AMBER] Tleap naming problem
Rui Chen (Wed, 1 Jan 2020 14:19:00 -0700):
Dear David,
 
 I am preparing the protein/ligand complex system to run MD simulation
 
according to the following tutorial:
 
 http://ambermd.org/tutorials/basic/tutorial4b/  .
 
 I order to make the Tleap u...
------------------------------------------------------------
[Neha Gandhi] [AMBER] ligand clustering
Neha Gandhi (Thu, 2 Jan 2020 15:47:51 +1000):
Dear List,
 
 I have a system consisting of ligand, protein, solvent and ions. The ligand
 
is initially positioned randomly in the solvent. I am interested in
 
performing cluster analysis in order to id...
------------------------------------------------------------
[Sruthi Sudhakar] [AMBER] Regarding clustering using mmtsb toolkit
Sruthi Sudhakar (Thu, 2 Jan 2020 12:07:22 +0530):
Dear all,
 
 I would like to know if there is an alternative software or methods to do
 
the clustering in AMBER. Right now, I am doing the tut

Choose an option (1-5):  5


Exiting.
